# Field Shift · Punto 2: predecir el vigor de cada cultivo

**Objetivo final:** un modelo que, para un punto de un campo, prediga qué tan bien crecería cada cultivo (`ndvi_peak`) según su historial, el suelo y el clima. Con eso recomendamos el cultivo y mostramos una banda de riesgo.

**Problema:** `ndvi_peak` todavía no llegó (lo trae el paquete de MODIS). Por eso la notebook tiene dos partes:

- **Parte A, se corre hoy:** un modelo de práctica que predice *qué cultivo sembró el productor* (`crop_t`). Sirve para armar todo el pipeline: cargar datos, preparar variables, separar por años, entrenar y medir por región.
- **Parte B, se activa sola cuando llegue `ndvi_peak`:** el modelo de vigor de verdad, la función que recomienda cultivos y la simulación con muchos climas (Monte Carlo).

Cómo usarla: abrila desde cualquier carpeta del repo y corré las celdas de arriba hacia abajo (Shift + Enter). Los archivos se buscan a partir de la raíz del repo, así que no hace falta moverlos.

## 0. Instalar y cargar librerías

- **pandas:** maneja tablas, como un Excel desde Python.
- **numpy:** cuentas con números.
- **LightGBM:** el modelo. Arma cientos de árboles de decisión chiquitos, cada uno corrigiendo los errores del anterior ("gradient boosting"). Funciona muy bien con tablas.

Si no tenés LightGBM instalado, sacale el `#` a la línea de abajo y corré la celda una vez. Si igual no se puede instalar, la notebook usa un modelo equivalente de scikit-learn.

In [1]:
#!pip install lightgbm

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

try:
    import lightgbm as lgb
    USAR_LGBM = True
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
    USAR_LGBM = False

print("LightGBM disponible:", USAR_LGBM)

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


LightGBM disponible: True


## 1. Cargar los datos

Cada fila es **un punto de un campo en un año**. Columnas importantes:

- `crop_t`: cultivo de ese año.
- `crop_t1`, `crop_t2`: cultivo de 1 y 2 años antes.
- `n_maiz_prev`, `n_soja_prev`, `n_trigo_prev`: cuántas veces se sembró cada uno antes.
- `region`: código del condado. Se lee como texto para no perder ceros.

In [ ]:
from pathlib import Path

# Raíz del repo: la primera carpeta hacia arriba que tenga .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
RUTA_CSV = RAIZ / "cdl_puntos.csv"

df = pd.read_csv(RUTA_CSV, dtype={"region": str, "point_id": str})

NOMBRES_REGION = {"19169": "Story (IA)", "17113": "McLean (IL)", "20191": "Sumner (KS)"}
df["region_nombre"] = df["region"].map(NOMBRES_REGION)

print("Filas y columnas:", df.shape)
df.head()

## 2. Mirar los datos antes de modelar

Porcentaje de cada cultivo por región. Fijate que en Iowa e Illinois es casi todo maíz y soja, y que en Kansas hay mucho trigo. Esto se llama **desbalance de clases**: el modelo va a ver muchísimos ejemplos de algunos cultivos y casi ninguno de otros.

In [4]:
pd.crosstab(df["region_nombre"], df["crop_t"], normalize="index").mul(100).round(1)

crop_t,alfalfa,barbecho,doble_trigo_soja,maiz,otros_cultivos,otros_dobles,soja,sorgo,trigo
region_nombre,,,,,,,,,
McLean (IL),0.0,0.1,0.0,50.4,0.1,0.0,49.2,0.0,0.2
Story (IA),0.3,0.0,0.0,57.9,0.1,0.0,41.6,0.0,0.0
Sumner (KS),0.6,0.6,14.6,10.0,7.1,1.0,20.9,4.8,40.4


## 3. Preparar las variables

1. **Agrupar cultivos raros** en `otros`. Con tan pocos ejemplos, el modelo no puede aprender nada confiable sobre ellos.
2. **Pasar los conteos a proporciones.** Haber tenido maíz 3 veces no es lo mismo en 4 años de historia que en 10, así que dividimos por los años disponibles.
3. Marcar las columnas de texto como **categóricas**, para que el modelo sepa que "maiz" y "soja" son categorías y no números.

In [5]:
CLASES_RARAS = ["alfalfa", "barbecho", "otros_dobles", "otros_cultivos"]

def agrupar(cultivo):
    return "otros" if cultivo in CLASES_RARAS else cultivo

for col in ["crop_t", "crop_t1", "crop_t2"]:
    df[col] = df[col].map(agrupar)

for c in ["maiz", "soja", "trigo"]:
    df[f"frac_{c}_prev"] = df[f"n_{c}_prev"] / df["n_anios_prev"]

CULTIVOS = sorted(df["crop_t"].unique())
print("Cultivos después de agrupar:", CULTIVOS)

for col in ["crop_t1", "crop_t2"]:
    df[col] = pd.Categorical(df[col], categories=CULTIVOS)
df["cultivo"] = pd.Categorical(df["crop_t"], categories=CULTIVOS)  # crop_t como categoría (se usa en la Parte B)
df["region_cat"] = pd.Categorical(df["region"])

df["crop_t"].value_counts()

Cultivos después de agrupar: ['doble_trigo_soja', 'maiz', 'otros', 'soja', 'sorgo', 'trigo']


crop_t
maiz                31947
soja                30163
trigo               10979
doble_trigo_soja     3959
otros                2666
sorgo                1286
Name: count, dtype: int64

## 4. Split: separar por años

- **Train (2017–2023):** con esto aprende el modelo.
- **Validación (2024):** con esto comparamos modelos y ajustamos.
- **Test (2025):** se mira **una sola vez**, al final.

No mezclamos filas al azar: cada punto aparece 9 veces (una por año) y el modelo haría "trampa" reconociendo el campo. Separando por años, el test simula predecir un año que todavía no pasó.

In [6]:
train = df[df["year"] <= 2023]
valid = df[df["year"] == 2024]
test  = df[df["year"] == 2025]

print("train:", len(train), "| validación:", len(valid), "| test:", len(test))

train: 63000 | validación: 9000 | test: 9000


# Parte A · Modelo de práctica: predecir el cultivo sembrado

Esto **no** es la recomendación: predice lo que el productor *hace*, no lo que le *conviene*. Lo usamos para dejar el pipeline andando.

## 5. Baseline: la regla tonta a superar

"Lo contrario del año anterior": si el año pasado fue maíz, decimos soja, y al revés. Si fue otra cosa, decimos lo mismo que el año pasado. Si el modelo no le gana a esto, no aporta nada.

In [7]:
def regla_baseline(cultivo_anterior):
    if cultivo_anterior == "maiz":
        return "soja"
    if cultivo_anterior == "soja":
        return "maiz"
    return cultivo_anterior

def acierto_por_region(datos, predicciones):
    tabla = pd.DataFrame({
        "region": datos["region_nombre"].values,
        "acierto": (datos["crop_t"].values == np.asarray(predicciones)),
    })
    por_region = tabla.groupby("region")["acierto"].mean()
    por_region["TOTAL"] = tabla["acierto"].mean()
    return (por_region * 100).round(1)

pred_baseline_valid = valid["crop_t1"].astype(str).map(regla_baseline)
acierto_por_region(valid, pred_baseline_valid).to_frame("baseline_%")

,baseline_%
region,
McLean (IL),94.2
Story (IA),83.4
Sumner (KS),34.7
TOTAL,70.8


## 6. Entrenar el modelo

`FEATURES` es la lista de columnas que el modelo puede mirar. `fit` es el momento en que aprende.

In [8]:
FEATURES_A = ["crop_t1", "crop_t2", "frac_maiz_prev", "frac_soja_prev", "frac_trigo_prev",
              "region_cat", "lat", "lon"]

if USAR_LGBM:
    modelo_a = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1)
else:
    modelo_a = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                              categorical_features="from_dtype", random_state=0)

modelo_a.fit(train[FEATURES_A], train["crop_t"])
print("Listo, modelo entrenado.")

Listo, modelo entrenado.


## 7. Comparar contra el baseline (en validación, por región)

Se mira cada región por separado: un buen promedio general puede esconder que en Kansas anda mal.

In [9]:
pred_modelo_valid = modelo_a.predict(valid[FEATURES_A])

comparacion = pd.DataFrame({
    "baseline_%": acierto_por_region(valid, pred_baseline_valid),
    "modelo_%":   acierto_por_region(valid, pred_modelo_valid),
})
comparacion["mejora_pts"] = comparacion["modelo_%"] - comparacion["baseline_%"]
comparacion

,baseline_%,modelo_%,mejora_pts
region,,,
McLean (IL),94.2,94.9,0.7
Story (IA),83.4,85.9,2.5
Sumner (KS),34.7,55.0,20.3
TOTAL,70.8,78.6,7.8


## 8. Examen final (test 2025)

Por eso la celda está apagada: cambiá `MIRAR_TEST` a `True` **solo cuando ya no vayas a cambiar nada del modelo**. Si después de ver el test volvés a ajustar, el test deja de ser un examen honesto.

Las notas de Lucho avisan que el CDL 2025 viene a otra resolución (10 m, llevado a 30 m), así que puede dar un poco distinto que 2024.

In [ ]:
MIRAR_TEST = False   # pasalo a True una sola vez, al final

if MIRAR_TEST:
    display(pd.DataFrame({
        "baseline_%": acierto_por_region(test, test["crop_t1"].astype(str).map(regla_baseline)),
        "modelo_%":   acierto_por_region(test, modelo_a.predict(test[FEATURES_A])),
    }))
else:
    print("Test 2025 sin mirar todavía (MIRAR_TEST = False).")

# Parte B · Modelo de vigor (se activa cuando llegue `ndvi_peak`)

La idea:

```
f(cultivo, cultivo_año_anterior, cultivo_hace_2_años, proporciones, suelo, clima) -> ndvi_peak
```

Para **recomendar**, tomamos un punto y le preguntamos al modelo: "¿y si acá pusieras maíz? ¿y soja? ¿y trigo?". Cambiamos solo el cultivo, dejamos todo lo demás igual y comparamos las predicciones.

Reglas de las notas de Lucho:

- Como variable de entrada, **solo datos de antes de la siembra** (clima de abril–mayo, años previos). El NDVI de la temporada es el resultado, nunca una entrada.
- Clave de unión con los otros datasets: `point_id` + `year`. Suelo, solo por `point_id`.

## 9. Unir los datos nuevos

Cuando el equipo suba los archivos, completá las rutas. Mientras estén en `None`, la Parte B no hace nada.

De cada archivo se traen **solo** las columnas listadas, así no se cuelan columnas repetidas (`lat`, `region`) ni variables de la temporada que filtren el resultado. Si un archivo tiene filas repetidas para la misma clave, la unión corta con error en vez de duplicar filas sin avisar.

In [ ]:
RUTA_NDVI  = None   # ej.: RAIZ / "data/processed/modis/modis_puntos.csv"   (columnas: point_id, year, ndvi_peak)
RUTA_CLIMA = None   # ej.: RAIZ / "data/processed/power/power_puntos.csv"   (columnas: point_id, year, ...)
RUTA_SUELO = None   # ej.: RAIZ / "data/processed/ssurgo/ssurgo_puntos.csv" (columnas: point_id, ...)

TARGET = "ndvi_peak"
COLUMNAS_CLIMA = []   # ej.: ["prec_abr_may_mm", "sm_siembra_m3m3", "tmax_jul_c"]
COLUMNAS_SUELO = []   # ej.: ["arcilla_pct", "materia_organica_pct", "pendiente_pct"]

def unir(base, ruta, claves, columnas):
    nuevo = pd.read_csv(ruta, dtype={"point_id": str}, usecols=claves + columnas)
    return base.merge(nuevo, on=claves, how="left", validate="many_to_one")

dfv = df.copy()
if RUTA_NDVI:
    dfv = unir(dfv, RUTA_NDVI, ["point_id", "year"], [TARGET])
if RUTA_CLIMA:
    dfv = unir(dfv, RUTA_CLIMA, ["point_id", "year"], COLUMNAS_CLIMA)
if RUTA_SUELO:
    dfv = unir(dfv, RUTA_SUELO, ["point_id"], COLUMNAS_SUELO)

HAY_TARGET = TARGET in dfv.columns
print("¿Está ndvi_peak?", HAY_TARGET)

## 10. Entrenar el modelo de vigor

Ahora es una **regresión**: predecimos un número (el NDVI) en vez de una categoría. La métrica es el **MAE**, el error promedio en unidades de NDVI. Por ejemplo, un MAE de 0,03 quiere decir que en promedio le erra por 0,03.

**Baseline a superar:** el NDVI promedio de ese cultivo en esa región durante el entrenamiento. Si el modelo no le gana, el historial, el clima y el suelo no están aportando nada.

In [ ]:
FEATURES_B = ["cultivo", "crop_t1", "crop_t2", "frac_maiz_prev", "frac_soja_prev", "frac_trigo_prev",
              "region_cat", "lat", "lon"] + COLUMNAS_CLIMA + COLUMNAS_SUELO

modelo_b = None
if not HAY_TARGET:
    print("Todavía no hay ndvi_peak: la Parte B queda en espera.")
else:
    datos = dfv.dropna(subset=[TARGET])
    tr = datos[datos["year"] <= 2023]
    va = datos[datos["year"] == 2024]

    if USAR_LGBM:
        modelo_b = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31, verbose=-1)
    else:
        modelo_b = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.03,
                                                 categorical_features="from_dtype", random_state=0)
    modelo_b.fit(tr[FEATURES_B], tr[TARGET])

    # Baseline: media de entrenamiento por región y cultivo (si la combinación no existe, media de la región)
    media_rc = tr.groupby(["region", "crop_t"])[TARGET].mean()
    media_r  = tr.groupby("region")[TARGET].mean()
    pred_base = pd.Series([media_rc.get((r, c), media_r[r]) for r, c in zip(va["region"], va["crop_t"])],
                          index=va.index)

    mae = pd.DataFrame({
        "baseline_mae": (va[TARGET] - pred_base).abs(),
        "modelo_mae":   (va[TARGET] - modelo_b.predict(va[FEATURES_B])).abs(),
        "region":       va["region_nombre"],
    })
    tabla = mae.groupby("region")[["baseline_mae", "modelo_mae"]].mean()
    tabla.loc["TOTAL"] = mae[["baseline_mae", "modelo_mae"]].mean()
    print("MAE en validación (2024), más bajo es mejor:")
    display(tabla.round(4))

## 11. Recomendar: probar cada cultivo en un punto

Solo recomendamos cultivos con datos suficientes **en esa región**. En Iowa e Illinois casi no hay trigo ni sorgo, así que el modelo estaría adivinando: ahí solo comparamos maíz y soja.

In [13]:
CANDIDATOS_POR_REGION = {
    "19169": ["maiz", "soja"],
    "17113": ["maiz", "soja"],
    "20191": ["maiz", "soja", "trigo", "sorgo", "doble_trigo_soja"],
}

def recomendar(fila, modelo):
    """fila: una fila de dfv (un punto en un año). Devuelve el NDVI esperado para cada cultivo candidato."""
    candidatos = CANDIDATOS_POR_REGION[str(fila["region"])]
    escenarios = pd.DataFrame([fila] * len(candidatos))
    escenarios["cultivo"] = pd.Categorical(candidatos, categories=CULTIVOS)
    for col in ["crop_t1", "crop_t2"]:
        escenarios[col] = pd.Categorical(escenarios[col].astype(str), categories=CULTIVOS)
    escenarios["region_cat"] = pd.Categorical(escenarios["region"].astype(str), categories=df["region_cat"].cat.categories)
    escenarios["ndvi_esperado"] = modelo.predict(escenarios[FEATURES_B])
    return escenarios[["cultivo", "ndvi_esperado"]].sort_values("ndvi_esperado", ascending=False)

if modelo_b is not None:
    ejemplo = dfv[(dfv["year"] == 2024) & (dfv["region"] == "19169")].iloc[0]
    print("Punto:", ejemplo["point_id"], "| antes:", ejemplo["crop_t2"], "->", ejemplo["crop_t1"])
    display(recomendar(ejemplo, modelo_b))

## 12. Monte Carlo: el mismo punto con muchos climas

Para cada cultivo candidato probamos el clima de **todos los años históricos** de ese punto, y así obtenemos una distribución de resultados en vez de un solo número:

- **media:** qué esperar en un año normal.
- **p10:** cómo te va en un año malo (solo 1 de cada 10 años es peor).
- **p90:** cómo te va en un año muy bueno.

Esta es la salida que necesita el front: `point_id, cultivo, media, p10, p90`. Solo funciona cuando haya columnas de clima en `COLUMNAS_CLIMA`.

In [14]:
def simular_escenarios(fila, modelo, dfv):
    climas = dfv.loc[dfv["point_id"] == fila["point_id"], COLUMNAS_CLIMA].drop_duplicates()
    resultados = []
    for cultivo in CANDIDATOS_POR_REGION[str(fila["region"])]:
        escenarios = pd.DataFrame([fila] * len(climas))
        escenarios[COLUMNAS_CLIMA] = climas.values
        escenarios["cultivo"] = pd.Categorical([cultivo] * len(climas), categories=CULTIVOS)
        for col in ["crop_t1", "crop_t2"]:
            escenarios[col] = pd.Categorical(escenarios[col].astype(str), categories=CULTIVOS)
        escenarios["region_cat"] = pd.Categorical(escenarios["region"].astype(str), categories=df["region_cat"].cat.categories)
        pred = modelo.predict(escenarios[FEATURES_B])
        resultados.append({"point_id": fila["point_id"], "cultivo": cultivo,
                           "media": pred.mean(), "p10": np.percentile(pred, 10), "p90": np.percentile(pred, 90)})
    return pd.DataFrame(resultados).sort_values("media", ascending=False)

if modelo_b is not None and COLUMNAS_CLIMA:
    display(simular_escenarios(ejemplo, modelo_b, dfv))
else:
    print("Falta el modelo de vigor o las columnas de clima: queda en espera.")

Falta el modelo de vigor o las columnas de clima: queda en espera.


## Próximos pasos

- [ ] Correr la Parte A y anotar en el issue #6 cuánto le gana el modelo al baseline en cada región.
- [ ] Cruzar `FEATURES_B` con el catálogo de variables de Diego y anotar qué falta.
- [ ] Preguntar al equipo los nombres de archivo y columnas de MODIS (NDVI), POWER (clima) y SSURGO (suelo) para completar el paso 9.
- [ ] Cuando ande la Parte B, revisar los chequeos agronómicos de `ML_NOTAS_CDL.md` (sección 6). Por ejemplo: maíz después de soja tiene que dar mejor que maíz después de maíz.
- [ ] Punto 3 (explicaciones): usar SHAP sobre `modelo_b`.